Copyright 2026 DataRobot, Inc. and its affiliates.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

In [ ]:
from jointfm_client import bootstrap_notebook

bootstrap_notebook(add_src_root=True)

# Maximum Utilization
Fill every request slot of the configured JointFM deployment at once with the largest request its `/healthz` capacity envelope admits, and report a PASS/FAIL verdict with the wall-clock time. The request is sized from the `data_generation` block: `max_series` columns, an `n_input`-row history, an `n_output`-step horizon, and `max_sample_count` sample paths. Every column is a target, so the response is also the largest the deployment can be asked for. The history is a seeded synthetic random walk; only the request's size matters here, not its values.

Each reachable endpoint (every pool instance when `JOINTFM_DEPLOYMENT_IDS` configures a pool) advertises `max_concurrent_requests`, the number of requests it serves at the same time, and receives that many maximal requests simultaneously, each for its own `max_sample_count`. This is the one place the client occupies more than one slot per endpoint; ordinary forecasts send at most one request per endpoint at a time.

Each run spends real GPU time on the hosted deployment. The terminal equivalent is `uv run jointfm-client max-utilization`, which exits with status 1 on FAIL.

In [ ]:
from jointfm_client import JointFMClient, JointFMRetryConfig, run_max_utilization_check

# Retries would turn a failure at maximum load into a slower success.
client = JointFMClient.from_env(retry_config=JointFMRetryConfig(max_attempts=1))
report = run_max_utilization_check(client)
print("\n".join(report.format_lines()))

## Burst Timing
The burst fails when any request raises a JointFM service or transport error or returns the wrong sample count. An endpoint that fails the health probe fails the whole check.

`wallclock_s` spans request building, every round trip, and response parsing. It is therefore an upper bound on each single request's duration, and `read_timeout_share` relates that bound to the client's per-request read timeout. A share close to 1 means a slightly larger request, or a busier deployment, would time out. The client parses all concurrent responses in one Python process, so the time also grows with the number of slots when the deployment serves them in parallel.

In [ ]:
import pandas as pd

stage = report.concurrent_saturation
pd.DataFrame(
    [
        {
            "stage": stage.name,
            "passed": stage.passed,
            "n_samples": stage.n_samples,
            "requests": stage.request_count,
            "wallclock_s": round(stage.wallclock_seconds, 2),
            "read_timeout_share": round(
                stage.wallclock_seconds / report.read_timeout_seconds, 3
            ),
            "errors": "; ".join(stage.errors),
        }
    ]
)

In [ ]:
if not report.passed:
    raise RuntimeError("JointFM maximum-utilization check failed; see the report above")
print("JointFM maximum-utilization check passed")